# Figures 2, 3, 9

Reproduces the cross-domain framing figure (paper **Figure 2**), the hiring content-vs-delivery by-accent figure (paper **Figure 3**), and the all-domains stacked figure (paper **Figure 9**), across the four focus models and the four domains (hiring human/synthetic, presentation, english-tests).

## Setup

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mc
from matplotlib.patches import Patch
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi']  = 120
plt.rcParams['savefig.dpi'] = 150

ROOT = Path('..').resolve()
FIGS = ROOT / 'results' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

PAPER_ACCENTS = ['American', 'British', 'Indian', 'Chinese', 'Nigerian']
NATIVE        = ['American', 'British']
NON_NATIVE    = ['Indian', 'Chinese', 'Nigerian']

MODEL_DISPLAY = {
    'Gemini-2.5-Flash': 'Gemini 2.5 Flash',
    'Gemini-2.5-Pro':   'Gemini 2.5 Pro',
    'Qwen':             'Qwen3-Omni',
    'GPT-Audio-1.5':    'GPT-Audio-1.5',
}
FOCUS_MODELS = ['Gemini-2.5-Flash', 'Gemini-2.5-Pro', 'Qwen', 'GPT-Audio-1.5']
MODEL_FILE = {'Gemini-2.5-Flash': 'gemini-2.5-flash', 'Gemini-2.5-Pro': 'gemini-2.5-pro',
              'Qwen': 'qwen', 'GPT-Audio-1.5': 'gpt-audio-1.5'}

PROMPT_ORDER = ['critical', 'ideal', 'native']
PT_COLORS    = {'critical': '#e67e22', 'ideal': '#3498db', 'native': '#8e44ad'}

def _extract_rating(text):
    # v2 fix: case-insensitive + handles bare digit and JSON-style "rating": 5
    s = str(text).strip()
    if re.fullmatch(r'\d', s):
        return int(s)
    m = re.search(r'"?rating"?\s*[:=]\s*(\d)', s, re.IGNORECASE)
    return int(m.group(1)) if m else np.nan

def _normalize_accent(raw):
    raw = str(raw).strip()
    for a in PAPER_ACCENTS:
        if a.lower() in raw.lower():
            return a
    return None

def _zscore(df, groups, val='rating', out='z'):
    df = df.copy()
    g  = df.groupby(groups)[val]
    df[out] = (df[val] - g.transform('mean')) / g.transform('std')
    return df

def _compute_pt_avg(df, model_col='model_label', style_col='prompt_type'):
    """Per-(model, prompt_style) mean gap of non-native accents vs (American+British) baseline."""
    cell = df.groupby([model_col, style_col, 'accent'])['z'].mean().reset_index()
    base = (cell[cell['accent'].isin(NATIVE)]
            .groupby([model_col, style_col])['z'].mean())
    cell['gap'] = cell.apply(
        lambda r: r['z'] - base.get((r[model_col], r[style_col]), 0.0), axis=1)
    return (cell[~cell['accent'].isin(NATIVE)]
            .groupby([model_col, style_col])['gap'].mean().reset_index())

# --- Shared accent palette for the by-accent figures (Figures 3 & 9) ---
ACCENT_BASE_ALT = {'American': '#9B72CF', 'British': '#0072B2', 'Chinese': '#F0A500',
                   'Indian': '#009E73', 'Nigerian': '#D55E00'}
LIGHT_MIX = 0.78
HATCH_BY_TYPE = {'content': None, 'fluency': '///', 'pronunciation': 'xxx', 'delivery': None}

def _shade_alt(accent, idx, n):
    base = np.array(mc.to_rgb(ACCENT_BASE_ALT[accent]))
    light = base + (np.array([1.0, 1.0, 1.0]) - base) * LIGHT_MIX
    if n <= 1:
        return tuple(base)
    return tuple(light + (base - light) * (idx / (n - 1)))


In [ ]:
CORPUS_FILES = {
    'Gemini-2.5-Flash': 'gemini-2.5-flash_hiring_corpus.csv',
    'Gemini-2.5-Pro':   'gemini-2.5-pro_hiring_corpus.csv',
    'Qwen':             'qwen_hiring_corpus.csv',
    'GPT-Audio-1.5':    'gpt-audio-1.5_hiring_corpus.csv',
}

corpus_full_parts = []
for mlabel, fname in CORPUS_FILES.items():
    p = ROOT / 'results' / 'hiring_corpus' / fname
    if not p.exists():
        print(f'WARNING: missing {fname}')
        continue
    df = pd.read_csv(p)
    df['model_label'] = mlabel
    corpus_full_parts.append(df)

corpus_full = pd.concat(corpus_full_parts, ignore_index=True)
corpus_full = corpus_full[corpus_full['model_label'].isin(FOCUS_MODELS)].copy()
corpus_full['rating'] = corpus_full['model_output'].apply(_extract_rating)
corpus_full['accent'] = corpus_full['accent_nationality_origin'].apply(_normalize_accent)
corpus_full = corpus_full[corpus_full['accent'].notna() & corpus_full['rating'].notna()].copy()
corpus_full = _zscore(corpus_full, ['model_label', 'prompt_type', 'script_type'])

# Hiring 'script_type==delivery' is the vocal delivery axis
hiring_del = corpus_full[corpus_full['script_type'] == 'delivery'].copy()
hiring_pt_avg = _compute_pt_avg(hiring_del, style_col='prompt_type')
print('Hiring rows:', len(hiring_del))
print('  by model:', hiring_del['model_label'].value_counts().to_dict())
print(hiring_pt_avg)

In [ ]:
HSYNTH_FILES = {
    ('Gemini-2.5-Flash', 'critical'): 'gemini-2.5-flash_hiring_synthetic_critical.csv',
    ('Gemini-2.5-Flash', 'ideal'):    'gemini-2.5-flash_hiring_synthetic_ideal.csv',
    ('Gemini-2.5-Flash', 'native'):   'gemini-2.5-flash_hiring_synthetic_native.csv',
    ('Gemini-2.5-Pro',   'critical'): 'gemini-2.5-pro_hiring_synthetic_critical.csv',
    ('Gemini-2.5-Pro',   'ideal'):    'gemini-2.5-pro_hiring_synthetic_ideal.csv',
    ('Gemini-2.5-Pro',   'native'):   'gemini-2.5-pro_hiring_synthetic_native.csv',
    ('Qwen',             'critical'): 'qwen_hiring_synthetic_critical.csv',
    ('Qwen',             'ideal'):    'qwen_hiring_synthetic_ideal.csv',
    ('Qwen',             'native'):   'qwen_hiring_synthetic_native.csv',
    ('GPT-Audio-1.5',    'critical'): 'gpt-audio-1.5_hiring_synthetic_critical.csv',
    ('GPT-Audio-1.5',    'ideal'):    'gpt-audio-1.5_hiring_synthetic_ideal.csv',
    ('GPT-Audio-1.5',    'native'):   'gpt-audio-1.5_hiring_synthetic_native.csv',
}

hsynth_parts = []
for (model, style), fname in HSYNTH_FILES.items():
    p = ROOT / 'results' / 'hiring_synthetic' / fname
    if not p.exists():
        print(f'WARNING: missing {fname}')
        continue
    df = pd.read_csv(p)
    df = df.drop(columns=['category.1', 'script_type.1'], errors='ignore')
    df['model_label']  = model
    df['prompt_style'] = style
    hsynth_parts.append(df)

hsynth = pd.concat(hsynth_parts, ignore_index=True)
hsynth['rating'] = hsynth['model_output'].apply(_extract_rating)
hsynth['accent'] = hsynth['accent'].apply(_normalize_accent)
hsynth = hsynth[hsynth['accent'].notna() & hsynth['rating'].notna()].copy()
hsynth = hsynth.drop_duplicates(subset=['model_label', 'prompt_style', 'script_type', 'accent', 'voice_id', 'category'])

# Same semantics as hiring corpus: script_type=='delivery' is vocal delivery
hsynth_del = hsynth[hsynth['script_type'] == 'delivery'].copy()
hsynth_del = _zscore(hsynth_del, ['model_label', 'prompt_style', 'script_type'])

hsynth_pt_avg = _compute_pt_avg(hsynth_del, style_col='prompt_style').rename(columns={'prompt_style': 'prompt_type'})
print('Hiring synthetic rows (delivery only):', len(hsynth_del))
print('  by model:', hsynth_del['model_label'].value_counts().to_dict())
print(hsynth_pt_avg)

In [ ]:
# Education results renamed: <model>_education_<prompt>_<context>.csv (results/education/)
EDU_FILES = {
    (m, style, setting): f"{MODEL_FILE[m]}_education_{style.replace('_v2','')}_{setting}.csv"
    for m in FOCUS_MODELS for style in ['critical_v2', 'ideal_v2'] for setting in ['classroom', 'conference']
}

edu_parts = []
for (model, style, setting), fname in EDU_FILES.items():
    p = ROOT / 'results' / 'education' / fname
    if not p.exists():
        print(f'WARNING: missing {fname}')
        continue
    df = pd.read_csv(p)
    df = df.drop(columns=['category.1', 'script_type.1'], errors='ignore')
    df = df.rename(columns={'category': 'prompt_category', 'script_type': 'prompt_type'})
    df['model_label']  = model
    df['prompt_style'] = style
    df['setting']      = setting
    edu_parts.append(df)

edu = pd.concat(edu_parts, ignore_index=True)
edu['rating']      = edu['model_output'].apply(_extract_rating)
edu['accent']      = edu['accent'].apply(_normalize_accent)
edu['script_kind'] = edu['script_filename'].apply(lambda fn: fn.split('_')[1])
edu['fluency']     = edu['script_filename'].apply(lambda fn: fn.split('_')[-1])

# Keep classroom-conceptual-fluent AND conference-presentation-fluent
edu = edu[edu['fluency'] == 'fluent'].copy()
edu = edu[((edu['setting']=='classroom')  & (edu['script_kind']=='conceptual')) |
          ((edu['setting']=='conference') & (edu['script_kind']=='presentation'))].copy()
edu = edu[edu['accent'].notna() & edu['rating'].notna()].copy()

# Labels in education are already SWAPPED to match hiring semantics:
# prompt_type=='delivery' is vocal delivery scoring (matches hiring 'delivery' axis)
edu_del = edu[edu['prompt_type'] == 'delivery'].copy()
# Pool classroom + conference; z-score within (model, prompt_style, prompt_type)
edu_del = _zscore(edu_del, ['model_label', 'prompt_style', 'prompt_type'])

edu_pt_avg = _compute_pt_avg(edu_del, style_col='prompt_style')
# Strip _v2 suffix so legend matches hiring (critical / ideal)
edu_pt_avg['prompt_type'] = edu_pt_avg['prompt_style'].str.replace('_v2', '', regex=False)
edu_pt_avg = edu_pt_avg[['model_label', 'prompt_type', 'gap']]
print('Education rows (classroom+conference pooled):', len(edu_del))
print('  by setting:', edu_del['setting'].value_counts().to_dict())
print(edu_pt_avg)

In [ ]:
IMM_FILES = {
    (m, s): f"{MODEL_FILE[m]}_immigration_{s}.csv"
    for m in FOCUS_MODELS for s in ['critical', 'native']
}

imm_parts = []
for (model, style), fname in IMM_FILES.items():
    p = ROOT / 'results' / 'immigration' / fname
    if not p.exists():
        print(f'WARNING: missing {fname}')
        continue
    df = pd.read_csv(p)
    df['model_label']  = model
    df['prompt_style'] = style
    imm_parts.append(df)

imm = pd.concat(imm_parts, ignore_index=True)
# Keep only the vocal-delivery prompt (matches hiring's delivery axis)
imm = imm[imm['script_type'] == 'delivery'].copy()
imm['rating'] = imm['model_output'].apply(_extract_rating)
imm['accent'] = imm['accent'].apply(_normalize_accent)
imm = imm[imm['accent'].notna() & imm['rating'].notna()].copy()

imm = _zscore(imm, ['model_label', 'prompt_style'])
imm_pt_avg = _compute_pt_avg(imm, style_col='prompt_style').rename(columns={'prompt_style': 'prompt_type'})
print('Immigration rows (delivery only):', len(imm))
print(imm_pt_avg)

In [ ]:
# Per-domain z-scored frames used by the by-accent figures (Figures 3 & 9).
hh = _zscore(corpus_full.copy(), ['model_label', 'script_type'])          # hiring: human
hs = _zscore(hsynth.copy(),      ['model_label', 'script_type'])          # hiring: synthetic
ed = _zscore(edu.copy(),         ['model_label', 'prompt_type'])          # presentation (education)

# English tests (immigration): reload keeping all four PTE score types.
_im_parts = []
for (_m, _s), _fn in IMM_FILES.items():
    _p = ROOT / 'results' / 'immigration' / _fn
    if not _p.exists():
        continue
    _d = pd.read_csv(_p); _d['model_label'] = _m; _d['prompt_style'] = _s
    _im_parts.append(_d)
im_full = pd.concat(_im_parts, ignore_index=True)
im_full['rating'] = im_full['model_output'].apply(_extract_rating)
im_full['accent'] = im_full['accent'].apply(_normalize_accent)
im_full = im_full[im_full['accent'].notna() & im_full['rating'].notna()].copy()
im_full = _zscore(im_full, ['model_label', 'script_type'])


## Figure 2

_(labeled "Figure 3" in the original drafting notebook — it is **Figure 2** in the paper.)_

In [ ]:
panel_data = {
    'Hiring: Human':         hiring_pt_avg,
    'Hiring: Synthetic':     hsynth_pt_avg,
    'Presentation: Synthetic':  edu_pt_avg,
    'English Tests: Synthetic': imm_pt_avg,
}
subtitles = list(panel_data.keys())
fig, axes = plt.subplots(1, 4, figsize=(14, 6.5), sharex=True, sharey=True)
y       = np.arange(len(FOCUS_MODELS))
bar_w = 0.20      # actual bar thickness
spacing = 0.26    # center-to-center distance between bars
offsets = [-spacing, 0, spacing]
XLIM = (-1.55, 0.5)
for j, (ax, title) in enumerate(zip(axes, subtitles)):
    data = panel_data[title]
    # alternating background bands per model
    for k in range(len(FOCUS_MODELS)):
        if k % 2 == 0:
            ax.axhspan(k - 0.5, k + 0.5, facecolor='#f0f0f0', alpha=0.9, zorder=0)
    if data is not None:
        for i, pt in enumerate(PROMPT_ORDER):
            vals = []
            for m in FOCUS_MODELS:
                row = data[(data['model_label'] == m) & (data['prompt_type'] == pt)]
                vals.append(float(row['gap'].iloc[0]) if len(row) else np.nan)
            bars = ax.barh(y + offsets[i], vals, bar_w,
               label=pt.capitalize(), color=PT_COLORS[pt],
               edgecolor='white', linewidth=0.7, alpha=0.88)
            pad = (XLIM[1] - XLIM[0]) * 0.008
            for bar, v in zip(bars, vals):
                if not np.isfinite(v):
                    continue
                ax.text(v + (-pad if v < 0 else pad),
                        bar.get_y() + bar.get_height() / 2,
                        f'{v:+.2f}',
                        va='center',
                        ha='right' if v < 0 else 'left',
                        fontsize=15)
    ax.axvline(0, color='#555', linewidth=0.9, linestyle='--', alpha=0.55)
    ax.set_yticks(y)
    ax.set_yticklabels([MODEL_DISPLAY[m] for m in FOCUS_MODELS], fontsize=12)
    ax.tick_params(labelsize=14, colors='#222', width=1.0)
    ax.spines[['top', 'right']].set_visible(False)
    for s in ax.spines.values():
        s.set_color('#222')
        s.set_linewidth(1.2)
    ax.grid(axis='x', alpha=0.25)
    ax.set_xlim(*XLIM)
    ax.set_title(title, fontsize=15)
fig.supxlabel('Score gap (Non-Western) - (Western)', fontsize=17, y=0.16)
#plt.suptitle('Scoring gaps between core anglosphere and other accents', fontsize=16, y=1.03)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Prompt', fontsize=17, framealpha=0.9, title_fontsize=17,
           loc='lower center', bbox_to_anchor=(0.5,0.95), ncol=len(labels))
fig.subplots_adjust(bottom=0.26, top=0.88, left=0.08, right=0.98, wspace=0.12)
fig.savefig(FIGS / 'fig1_framing_domains_diff_updated_v2.pdf', bbox_inches='tight', dpi=300)
plt.show()
print('Saved: fig1_prompt_sensitivity_all_check_v2.pdf')

## Figure 3

_(labeled "Figure 2" in the original drafting notebook — it is **Figure 3** in the paper.)_

In [ ]:
# === Hiring: Human (SCRIPTED only) — excludes file_path containing "unscripted" ===
# Big figure, tight subplot spacing, larger fonts.

scripted_corpus = corpus_full[~corpus_full['file_path'].astype(str).str.contains('unscripted', case=False, na=False)].copy()
scripted_corpus = _zscore(scripted_corpus, ['model_label', 'script_type'])

print('Scripted-only rows by model:')
print(scripted_corpus.groupby('model_label').size().to_dict())
print('Dropped (unscripted) rows:', len(corpus_full) - len(scripted_corpus))

DOMAIN_DF      = scripted_corpus
DOMAIN_STYPES  = ['content', 'delivery']
DOMAIN_ST_COL  = 'script_type'
DOMAIN_TITLE   = 'Hiring: Human (scripted only)'

models = FOCUS_MODELS
n_cols = len(models)
x      = np.arange(len(PAPER_ACCENTS))
n_st   = len(DOMAIN_STYPES)
bar_w  = 0.85 / n_st
use_hatch = n_st > 2

summary = (DOMAIN_DF.groupby(['model_label', 'accent', DOMAIN_ST_COL])['z']
                    .mean().reset_index())

# All fonts pushed higher; sharey=True + tight wspace gives them more room.
FS_TITLE   = 64   # fig.suptitle
FS_SUB     = 52   # axes title (model name)
FS_AXLBL   = 48   # ylabel
FS_TICK    = 46   # x + y tick labels
FS_LEG     = 46   # legend body
FS_LEGTTL  = 48   # legend title

with plt.rc_context({
    'axes.titlesize':        FS_SUB,
    'axes.labelsize':        FS_AXLBL,
    'xtick.labelsize':       FS_TICK,
    'ytick.labelsize':       FS_TICK,
    'legend.fontsize':       FS_LEG,
    'legend.title_fontsize': FS_LEGTTL,
    'figure.titlesize':      FS_TITLE,
}):
    fig, axes = plt.subplots(1, n_cols, figsize=(12.0 * n_cols, 15.5), sharey=True)
    if n_cols == 1:
        axes = [axes]

    for c, model in enumerate(models):
        ax  = axes[c]
        mdf = summary[summary['model_label'] == model]
        for i, accent in enumerate(PAPER_ACCENTS):
            for j, st in enumerate(DOMAIN_STYPES):
                row = mdf[(mdf['accent'] == accent) & (mdf[DOMAIN_ST_COL] == st)]['z']
                val = row.values[0] if len(row) else np.nan
                if not np.isfinite(val):
                    continue
                offset = (j - (n_st - 1) / 2) * bar_w
                hatch = HATCH_BY_TYPE.get(st) if use_hatch else None
                ax.bar(i + offset, val, bar_w,
                       color=_shade_alt(accent, j, n_st),
                       edgecolor='white', linewidth=0.8, hatch=hatch)
        ax.axhline(0, color='black', linewidth=1.0)
        ax.grid(axis='y', alpha=0.25)
        for spine in ax.spines.values():
            spine.set_edgecolor('#bbbbbb')
        ax.set_title(MODEL_DISPLAY.get(model, model))
        ax.set_xticks(x)
        ax.set_xticklabels(PAPER_ACCENTS, rotation=30, ha='right')

    axes[0].set_ylabel(f'{DOMAIN_TITLE}\n(z)')

    type_patches = [
        Patch(facecolor=str(0.85 - 0.55 * (j / (n_st - 1))),
              edgecolor='#444', label=st.capitalize(),
              hatch=HATCH_BY_TYPE.get(st))
        for j, st in enumerate(DOMAIN_STYPES)
    ]
    fig.legend(
        handles=type_patches, title='Score type', loc='lower center',
        bbox_to_anchor=(0.5, -0.12), ncol=len(DOMAIN_STYPES), frameon=True,
    )

    fig.suptitle(f'{DOMAIN_TITLE} — Content vs Delivery by Accent', y=1.02)

    # Manual layout — tight subplot spacing, no tight_layout (which would re-pad).
    fig.subplots_adjust(left=0.05, right=0.995, top=0.90, bottom=0.18, wspace=0.04)

    fig.savefig(FIGS / 'fig_hiring_human_scripted_only_alt_palette_v2.pdf', bbox_inches='tight')
    plt.show()
print('Saved: fig_hiring_human_scripted_only_alt_palette_v2.pdf')

## Figure 9

In [ ]:
# === All-domains stacked figure (cool/warm palette; palette helpers defined in Setup) ===
DOMAINS = [
    ('Hiring: Human',         hh,      ['content', 'delivery'],                            'script_type'),
    ('Hiring: Synthetic',     hs,      ['content', 'delivery'],                            'script_type'),
    ('Presentation: Synthetic',  ed,      ['content', 'delivery'],                            'prompt_type'),
    ('English Tests: Synthetic', im_full, ['content', 'fluency', 'pronunciation', 'delivery'], 'script_type'),
]

models = FOCUS_MODELS
n_rows = len(DOMAINS)
n_cols = len(models)
x      = np.arange(len(PAPER_ACCENTS))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.8 * n_cols, 3.2 * n_rows),
                          sharex=True, sharey='row')

for r, (name, df_dom, st_list, st_col) in enumerate(DOMAINS):
    n_st  = len(st_list)
    bar_w = 0.85 / n_st
    use_hatch = n_st > 2
    summary = (df_dom.groupby(['model_label', 'accent', st_col])['z']
                     .mean().reset_index())

    for c, model in enumerate(models):
        ax  = axes[r, c]
        mdf = summary[summary['model_label'] == model]

        for i, accent in enumerate(PAPER_ACCENTS):
            for j, st in enumerate(st_list):
                row = mdf[(mdf['accent'] == accent) & (mdf[st_col] == st)]['z']
                val = row.values[0] if len(row) else np.nan
                if not np.isfinite(val):
                    continue
                offset = (j - (n_st - 1) / 2) * bar_w
                hatch = HATCH_BY_TYPE.get(st) if use_hatch else None
                ax.bar(i + offset, val, bar_w,
                       color=_shade_alt(accent, j, n_st),
                       edgecolor='white', linewidth=0.4,
                       hatch=hatch)

        ax.axhline(0, color='black', linewidth=0.5)
        ax.grid(axis='y', alpha=0.25)
        for spine in ax.spines.values():
            spine.set_edgecolor('#cccccc')

        if r == 0:
            ax.set_title(MODEL_DISPLAY.get(model, model), fontsize=15)
        if r == n_rows - 1:
            ax.set_xticks(x)
            ax.set_xticklabels(PAPER_ACCENTS, fontsize=10, rotation=30, ha='right')
        else:
            ax.set_xticks(x)
            ax.set_xticklabels([])
        if c == 0:
            ax.set_ylabel(f'{name}\n(z)', fontsize=15)

# Two legends at the bottom: accent colors (left), score type shade+hatch (right)
accent_patches = [
    Patch(facecolor=ACCENT_BASE_ALT[acc], edgecolor='#444', label=acc)
    for acc in PAPER_ACCENTS
]
accent_legend = fig.legend(
    handles=accent_patches, title='Accent', loc='lower center',
    bbox_to_anchor=(0.28, -0.03), ncol=len(PAPER_ACCENTS),
    fontsize=13, title_fontsize=14, frameon=True,
)
fig.add_artist(accent_legend)  # keep this legend when adding the next one

ALL_TYPES = ['content', 'fluency', 'pronunciation', 'delivery']
type_patches = [
    Patch(facecolor=str(0.85 - 0.55 * (j / 3)),
          edgecolor='#444', label=ALL_TYPES[j].capitalize(),
          hatch=HATCH_BY_TYPE.get(ALL_TYPES[j]))
    for j in range(4)
]
fig.legend(
    handles=type_patches, title='Score type', loc='lower center',
    bbox_to_anchor=(0.78, -0.03), ncol=4,
    fontsize=13, title_fontsize=14, frameon=True,
)

fig.suptitle('Content and Delivery Penalty Differences Across High-Stakes Domains',
             fontsize=15, y=1.005)
plt.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(FIGS / 'fig_paired_bars_all_domains_alt_palette_v2.pdf', bbox_inches='tight')
plt.show()
print('Saved: fig_paired_bars_all_domains_alt_palette_v2.pdf')